### Imports


In [30]:
import os
import json
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from google import genai

In [31]:
from pathlib import Path

from pathlib import Path

DATA_PATH = Path("../../legal-dataset/acts/consumer-protection-act-2019/final")

VECTOR_STORE = Path("../data/vector_store")


EMBEDDING_MODEL = "all-MiniLM-L6-v2"

TOP_K = 5

### Loading the JSON files


In [32]:
import os
import json
from pathlib import Path
from typing import List, Dict


class JSONKnowledgeLoader:
    """
    Loads all JSON knowledge cards from a directory recursively.

    Expected folder structure:

    knowledge-cards/
    ├── authorities/
    ├── definitions/
    ├── obligations/
    ├── procedures/
    └── rights/

    Every JSON file is considered one knowledge card.
    """

    def __init__(self, data_path: str):
        """
        Parameters
        ----------
        data_path : str
            Path to the knowledge-cards folder.
        """

        self.data_path = Path(data_path)

        if not self.data_path.exists():
            raise FileNotFoundError(
                f"Knowledge directory not found: {self.data_path}"
            )

    def load(self) -> List[Dict]:
        """
        Load every JSON file recursively.

        Returns
        -------
        List[Dict]
            List containing all JSON objects.
        """

        knowledge_cards = []

        json_files = list(self.data_path.rglob("*.json"))

        print(f"Found {len(json_files)} JSON files.")

        for file in json_files:

            try:

                with open(file, "r", encoding="utf-8") as f:

                    data = json.load(f)

                    # Add useful metadata about the file
                    data["_file_name"] = file.name
                    data["_category"] = file.parent.name
                    data["_path"] = str(file)

                    knowledge_cards.append(data)

            except Exception as e:

                print(f"Could not read {file}")
                print(e)

        print(f"Successfully loaded {len(knowledge_cards)} knowledge cards.")

        return knowledge_cards
loader = JSONKnowledgeLoader(
    "../../legal-dataset/acts/consumer-protection-act-2019/final"
)

knowledge_cards = loader.load()

Found 3 JSON files.
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\search-augmentation.json
list indices must be integers or slices, not str
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\v1-statute.json
list indices must be integers or slices, not str
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\v2-knowledge-cards.json
list indices must be integers or slices, not str
Successfully loaded 0 knowledge cards.


In [33]:
loader = JSONKnowledgeLoader(
    "../../legal-dataset/acts/consumer-protection-act-2019/final"
)

knowledge_cards = loader.load()

Found 3 JSON files.
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\search-augmentation.json
list indices must be integers or slices, not str
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\v1-statute.json
list indices must be integers or slices, not str
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\v2-knowledge-cards.json
list indices must be integers or slices, not str
Successfully loaded 0 knowledge cards.


### Documents readability


In [34]:
from typing import List, Dict
from langchain_core.documents import Document


class DocumentBuilder:
    """
    Converts JSON knowledge cards into LangChain Documents.

    Each JSON file becomes one Document that is optimized
    for semantic retrieval.
    """

    def __init__(self):
        pass

    def build_documents(self, knowledge_cards: List[Dict]) -> List[Document]:
        """
        Convert a list of JSON knowledge cards into LangChain Documents.
        """

        documents = []

        for card in knowledge_cards:

            document = self._build_single_document(card)

            documents.append(document)

        print(f"Created {len(documents)} LangChain Documents.")

        return documents

    def _build_single_document(self, card: Dict) -> Document:
        """
        Convert one JSON knowledge card into one LangChain Document.
        """

        content = card.get("content", {})
        search = card.get("search", {})
        metadata = card.get("metadata", {})

        # -----------------------------
        # Build human-readable text
        # -----------------------------

        text_parts = []

        # Title
        if card.get("title"):
            text_parts.append(f"Title: {card['title']}")

        # Legal Term
        if content.get("term"):
            text_parts.append(f"Term: {content['term']}")

        # Legal Definition
        if content.get("legal_definition"):
            text_parts.append(
                f"Legal Definition: {content['legal_definition']}"
            )

        # Plain Language
        if content.get("plain_language"):
            text_parts.append(
                f"Plain Language: {content['plain_language']}"
            )

        # Examples
        examples = content.get("examples", [])

        if examples:
            text_parts.append("Examples:")

            for example in examples:
                text_parts.append(f"- {example}")

        # Non Examples
        non_examples = content.get("non_examples", [])

        if non_examples:
            text_parts.append("Non Examples:")

            for example in non_examples:
                text_parts.append(f"- {example}")

        # Keywords
        keywords = search.get("keywords", [])

        if keywords:
            text_parts.append(
                "Keywords: " + ", ".join(keywords)
            )

        # Aliases
        aliases = search.get("aliases", [])

        if aliases:
            text_parts.append(
                "Aliases: " + ", ".join(aliases)
            )

        # Example User Questions
        user_queries = search.get("user_queries", [])

        if user_queries:
            text_parts.append("Possible User Questions:")

            for q in user_queries:
                text_parts.append(f"- {q}")

        page_content = "\n".join(text_parts)

        # -----------------------------
        # Metadata
        # -----------------------------

        doc_metadata = {

            "concept_id": card.get("concept_id"),

            "concept_type": card.get("concept_type"),

            "title": card.get("title"),

            "category": card.get("_category"),

            "file_name": card.get("_file_name"),

            "source_path": card.get("_path"),

            "jurisdiction": metadata.get("jurisdiction"),

            "act": metadata.get("act"),

            "language": metadata.get("language"),

            "confidence": metadata.get("confidence"),

            "version": metadata.get("version")
        }

        return Document(
            page_content=page_content,
            metadata=doc_metadata
        )

In [19]:
builder=DocumentBuilder()

documents=builder.build_documents(knowledge_cards)
documents

Created 195 LangChain Documents.


[Document(metadata={'concept_id': 'definition.advertisement', 'concept_type': 'definition', 'title': 'Advertisement', 'category': 'definitions', 'file_name': 'definition.advertisement.json', 'source_path': '..\\..\\legal-dataset\\acts\\consumer-protection-act-2019\\v2-knowledge-cards\\tier-a-reviewed\\definitions\\definition.advertisement.json', 'jurisdiction': 'India', 'act': 'Consumer Protection Act, 2019', 'language': 'en', 'confidence': 1.0, 'version': 1}, page_content='Title: Advertisement\nTerm: advertisement\nLegal Definition: any audio or visual publicity, representation, endorsement or pronouncement made by means of light, sound, smoke, gas, print, electronic media, internet or website and includes any notice, circular, label, wrapper, invoice or such other documents\nPlain Language: a public announcement or promotion using various media like audio, video, print, internet, etc.\nExamples:\n- A company running a TV commercial to promote its new product\n- A social media post en

### Embeddings


In [35]:
from typing import List

from langchain_huggingface import HuggingFaceEmbeddings


class EmbeddingManager:
    """
    Handles loading and using the embedding model.
    """

    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    ):
        """
        Parameters
        ----------
        model_name : str
            HuggingFace embedding model.
        """

        self.model_name = model_name
        self.embedding_model = self._load_model()

    def _load_model(self):
        """
        Load the HuggingFace embedding model.
        """

        print(f"Loading embedding model: {self.model_name}")

        model = HuggingFaceEmbeddings(
            model_name=self.model_name,
            model_kwargs={
                "device": "cpu"
            },
            encode_kwargs={
                "normalize_embeddings": True
            }
        )

        print("Embedding model loaded successfully.")

        return model

    def get_model(self):
        """
        Return the LangChain embedding model.
        """

        return self.embedding_model

    def embed_documents(self, texts: List[str]):
        """
        Generate embeddings for multiple documents.
        """

        return self.embedding_model.embed_documents(texts)

    def embed_query(self, query: str):
        """
        Generate embedding for a single query.
        """

        return self.embedding_model.embed_query(query)

In [21]:
embedding_manager=EmbeddingManager()

texts = [doc.page_content for doc in documents]

vectors = embedding_manager.embed_documents(texts)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3566.14it/s]


Embedding model loaded successfully.


### Vector Store and DB


In [36]:
from pathlib import Path
from typing import List

from langchain_core.documents import Document
from langchain_chroma import Chroma


class VectorStoreManager:
    """
    Handles creation, loading and persistence of the Chroma vector database.
    """

    def __init__(
        self,
        embedding_manager,
        persist_directory: str = "../data/vector_store",
        collection_name: str = "legal_knowledge_base"
    ):

        self.embedding_manager = embedding_manager

        self.embedding_model = embedding_manager.get_model()

        self.persist_directory = Path(persist_directory)

        self.collection_name = collection_name

        self.vectorstore = None

    # -------------------------------------------------------
    # Create Vector Store
    # -------------------------------------------------------

    def create_vector_store(self, documents: List[Document]) -> Chroma:
        """
        Create a new Chroma vector database from documents.
        """

        print(f"Creating Chroma Collection: {self.collection_name}")

        self.vectorstore = Chroma.from_documents(

            documents=documents,

            embedding=self.embedding_model,

            persist_directory=str(self.persist_directory),

            collection_name=self.collection_name

        )

        print(f"Stored {len(documents)} documents.")

        print(f"Database saved at {self.persist_directory}")

        return self.vectorstore

    # -------------------------------------------------------
    # Load Existing Vector Store
    # -------------------------------------------------------

    def load_vector_store(self) -> Chroma:
        """
        Load an existing Chroma vector database.
        """

        print(f"Loading Chroma Collection: {self.collection_name}")

        self.vectorstore = Chroma(

            collection_name=self.collection_name,

            embedding_function=self.embedding_model,

            persist_directory=str(self.persist_directory)

        )

        print("Vector Store Loaded Successfully.")

        return self.vectorstore

    # -------------------------------------------------------
    # Get Vector Store
    # -------------------------------------------------------

    def get_vector_store(self):

        if self.vectorstore is None:

            raise ValueError(
                "Vector Store not initialized."
            )

        return self.vectorstore

    # -------------------------------------------------------
    # Number of Stored Documents
    # -------------------------------------------------------

    def count_documents(self):

        if self.vectorstore is None:

            return 0

        return self.vectorstore._collection.count()

    # -------------------------------------------------------
    # Delete Collection
    # -------------------------------------------------------

    def delete_collection(self):

        if self.vectorstore is not None:

            self.vectorstore.delete_collection()

            print("Collection Deleted.")

            self.vectorstore = None

In [38]:
loader = JSONKnowledgeLoader(
    "../../legal-dataset/acts/consumer-protection-act-2019/final/"
)



knowledge_cards = loader.load()

builder = DocumentBuilder()

documents = builder.build_documents(knowledge_cards)

embedding_manager = EmbeddingManager()

vector_manager = VectorStoreManager(
    embedding_manager
)

vector_manager.create_vector_store(documents)

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


Found 3 JSON files.
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\search-augmentation.json
list indices must be integers or slices, not str
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\v1-statute.json
list indices must be integers or slices, not str
Could not read ..\..\legal-dataset\acts\consumer-protection-act-2019\final\v2-knowledge-cards.json
list indices must be integers or slices, not str
Successfully loaded 0 knowledge cards.
Created 0 LangChain Documents.
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


RuntimeError: Cannot send a request, as the client has been closed.

In [25]:
from typing import List

from langchain_core.documents import Document


class LegalRetriever:
    """
    Handles semantic retrieval from the Chroma Vector Database.
    """

    def __init__(
        self,
        vector_store_manager,
        k: int = 5
    ):

        self.vector_store = vector_store_manager.get_vector_store()

        self.k = k

        self.retriever = self.vector_store.as_retriever(

            search_type="similarity",

            search_kwargs={
                "k": self.k
            }

        )

    # -----------------------------------------------------
    # Retrieve Relevant Documents
    # -----------------------------------------------------

    def retrieve(
        self,
        query: str
    ) -> List[Document]:
        """
        Returns the Top-K most relevant documents.
        """

        documents = self.retriever.invoke(query)

        return documents

    # -----------------------------------------------------
    # Retrieve Context
    # -----------------------------------------------------

    def retrieve_context(
        self,
        query: str
    ) -> str:
        """
        Returns retrieved documents as one context string.
        """

        docs = self.retrieve(query)

        context = "\n\n".join(
            doc.page_content
            for doc in docs
        )

        return context

    # -----------------------------------------------------
    # Retrieve Sources
    # -----------------------------------------------------

    def retrieve_sources(
        self,
        query: str
    ):
        """
        Returns useful metadata for citations.
        """

        docs = self.retrieve(query)

        sources = []

        for doc in docs:

            sources.append({

                "title": doc.metadata.get("title"),

                "category": doc.metadata.get("category"),

                "concept_id": doc.metadata.get("concept_id"),

                "act": doc.metadata.get("act")

            })

        return sources

    # -----------------------------------------------------
    # Debug Search
    # -----------------------------------------------------

    def debug_search(
        self,
        query: str
    ):

        docs = self.retrieve(query)

        print(f"\nQuery : {query}")

        print(f"Retrieved {len(docs)} Documents\n")

        for i, doc in enumerate(docs, start=1):

            print("=" * 60)

            print(f"Result {i}")

            print("Title :", doc.metadata.get("title"))

            print("Category :", doc.metadata.get("category"))

            print()

            print(doc.page_content[:300])

            print()

In [26]:
retriever = LegalRetriever(
    vector_manager,
    k=5
)

In [27]:
docs = retriever.retrieve(
    "What is Consumer Right?"
)
docs

retriever.debug_search(
    "What is misleading advertisement?"
)


Query : What is misleading advertisement?
Retrieved 5 Documents

Result 1
Title : Misleading Advertisement
Category : definitions

Title: Misleading Advertisement
Term: misleading advertisement
Legal Definition: an advertisement, which— (i) falsely describes such product or service; or (ii) gives a false guarantee to, or is likely to mislead the consumers as to the nature, substance, quantity or quality of such product or servi

Result 2
Title : Misleading advertisement
Category : offences

Title: Misleading advertisement
Keywords: misleading, advertisement, manufacturer, service, provider, who, makes, causes, false, made
Aliases: Deceptive Advertising, False Ad, Fraudulent Claim, Misleading Promo, Untrue Advertisement, deceptive advertising, false ad
Possible User Questions:
- What is

Result 3
Title : Penalty for misleading advertisement
Category : penalties

Title: Penalty for misleading advertisement
Keywords: misleading advertisement, penalty, misleading, advertisement, publishin